# Part 3 · Waypoint demo (L7 with agentgateway)

**The story for the customer:** some controls only make sense at HTTP level, validating a user's JWT, allowing `GET` but not `DELETE`, splitting versions, rate-limiting. Those need a proxy in the path. In ambient that proxy is a **waypoint**, and on Solo Enterprise the waypoint is **agentgateway**, not Envoy.

This part adds the waypoint to the same petshop and layers on JWT authorisation, canary routing and identity-keyed rate limiting. It builds on Part 2's app, so if you are jumping straight here, run **Connect**, the **Part 2 context** cell, and **§2.1** (deploy petshop) first.

The division of labour stays clean: ztunnel keeps proving the caller's SPIFFE identity at L4, then hands the connection to agentgateway for L7, and that proven identity rides along as `source.identity.*` for every policy.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">ztunnel proves identity at L4, the waypoint enforces HTTP policy at L7</text> <defs> <marker id="wg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- pipeline --> <rect x="16" y="66" width="104" height="52" rx="8" fill="#eef2ff" stroke="#6366f1"/> <text x="68" y="88" text-anchor="middle" font-size="11.5" fill="#312e81">caller</text> <text x="68" y="104" text-anchor="middle" font-size="10" fill="#4338ca">+ user JWT</text> <line x1="120" y1="92" x2="148" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="148" y="66" width="150" height="52" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="223" y="87" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">ztunnel  ·  L4</text> <text x="223" y="104" text-anchor="middle" font-size="10" fill="#475569">proves SPIFFE identity</text> <line x1="298" y1="92" x2="326" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="326" y="60" width="214" height="64" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/> <text x="433" y="83" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">waypoint (agentgateway)  ·  L7</text> <text x="433" y="100" text-anchor="middle" font-size="10" fill="#166534">JWT authN + CEL authZ</text> <text x="433" y="114" text-anchor="middle" font-size="9.5" fill="#166534">identity rides along as source.identity.*</text> <line x1="540" y1="92" x2="568" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="568" y="66" width="120" height="52" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="628" y="96" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <!-- outcome pills --> <rect x="30" y="168" width="150" height="34" rx="17" fill="#fee2e2" stroke="#dc2626"/> <text x="105" y="190" text-anchor="middle" font-size="11" fill="#7f1d1d">no token → 401</text> <rect x="195" y="168" width="150" height="34" rx="17" fill="#dcfce7" stroke="#16a34a"/> <text x="270" y="190" text-anchor="middle" font-size="11" fill="#14532d">GET → 200</text> <rect x="360" y="168" width="150" height="34" rx="17" fill="#dcfce7" stroke="#16a34a"/> <text x="435" y="190" text-anchor="middle" font-size="11" fill="#14532d">DELETE (admin) → 200</text> <rect x="525" y="168" width="165" height="34" rx="17" fill="#fef3c7" stroke="#d97706"/> <text x="607" y="190" text-anchor="middle" font-size="11" fill="#7c2d12">DELETE (user) → 403</text> <text x="360" y="230" text-anchor="middle" font-size="11" fill="#64748b">Any valid token may read; only an admin may delete. One waypoint does authN, authZ, routing and rate limiting.</text> </svg></div>

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect**, then **Reset**, then the steps top to bottom.
> It needs `./setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

### Connect

Sets the contexts, the Solo `istioctl` build and your licence env, and confirms the platform is up. **Run this first, always.**

In [ ]:
# Connect — context, image/chart versions, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel >/dev/null 2>&1 \
  && echo "ambient mesh: up on ${CTX#kind-}" \
  || echo "mesh not found on ${CTX#kind-} — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"

## Reset + deploy · run this to (re)start the demo

Clears the waypoint and its L7 policies, then deploys the petshop this demo runs on, so this notebook stands alone (it does not need Part 2 to have been run). Safe on a fresh cluster.

In [ ]:
# Reset + deploy — clean slate, then (re)deploy the petshop this demo runs on.
# Safe on a fresh cluster: the deletes are no-ops, the deploy is idempotent.
: "${CTX:=kind-mesh1}"
kubectl --context $CTX -n petshop delete enterpriseagentgatewaypolicy \
  petshop-jwt petshop-authz petshop-ratelimit-storefront --ignore-not-found 2>/dev/null || true
kubectl --context $CTX -n petshop delete httproute petstore-split --ignore-not-found 2>/dev/null || true
kubectl --context $CTX -n petshop delete deploy petstore-v2 --ignore-not-found 2>/dev/null || true
kubectl --context $CTX -n petshop delete svc petstore-v2 --ignore-not-found 2>/dev/null || true
kubectl --context $CTX label namespace petshop istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CTX -n petshop delete gateway petshop-waypoint --ignore-not-found 2>/dev/null || true
# any L4 policy from Part 2 would mask the L7 story — clear those too
kubectl --context $CTX -n petshop delete authorizationpolicy \
  allow-storefront allow-checkout allow-gold-checkout --ignore-not-found 2>/dev/null || true

# the app this demo needs (same manifests as Part 2 §2.1)
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF
kubectl --context $CTX apply -f yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status \
  deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
echo "  ✓ clean + petshop deployed — run the waypoint steps below"

## 3.1 · Add the waypoint

**What we're doing:** put an agentgateway waypoint in front of the petshop so HTTP-level policy can run.

**How:** create a `Gateway` of class `enterprise-agentgateway-waypoint` and enrol the namespace onto it with one label. `setup.sh` already installed the control plane, so this just creates the waypoint and points the namespace at it.

**What you'll see:** the waypoint pod come up and its own request log showing the petshop traffic now flowing through it. We reset the L4 policies first so the L7 story stands on its own (in production you would keep both, defence in depth).

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# reset the L4 policies for a clean L7 story (incl. the claims policy)
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout --ignore-not-found

# the waypoint: a Gateway of class enterprise-agentgateway-waypoint …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petshop-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
# … then enrol the whole petshop NAMESPACE onto it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petshop-waypoint --overwrite
sleep 5
kubectl --context $CTX -n petshop rollout status deploy/petshop-waypoint --timeout=150s
# proof it is agentgateway: watch its own request log as the loops flow through
kubectl --context $CTX -n petshop logs deploy/petshop-waypoint --tail=2

## 3.2 · Authorise on the user's JWT

**What we're doing:** add user-level authorisation on top of workload identity: any signed-in user can read, only an admin can delete.

**How:** Keycloak (from `setup.sh`, realm `petshop`, users **alice**/`user` and **bob**/`admin`) issues the tokens. Two `EnterpriseAgentgatewayPolicy` objects sit on the waypoint. `jwtAuthentication` in Strict mode validates every request against Keycloak's JWKS, so no token means `401`. `authorization` then decides with CEL over the claims: any valid token may `GET`, and `DELETE` additionally needs `admin` in `realm_access.roles`.

**What you'll see:** no token → `401`, alice `GET` → `200`, alice `DELETE` → `403`, bob `DELETE` → `200`.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=300s
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    jwtAuthentication:
      mode: Strict
      providers:
        - issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          jwks:
            remote:
              backendRef: { name: keycloak, namespace: keycloak, port: 8080 }
              jwksPath: /realms/petshop/protocol/openid-connect/certs
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-authz, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    authorization:
      action: Allow
      policy:
        matchExpressions:
          - 'request.method == "GET"'
          - 'request.method == "DELETE" && "admin" in jwt.realm_access.roles'
EOF
sleep 10

# ── verify: both policies are attached to the waypoint ──
kubectl --context $CTX -n petshop get enterpriseagentgatewaypolicy petshop-jwt petshop-authz

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# the matrix: no token, alice (user), bob (admin).
# agentgateway answers a MISSING/invalid token with 401 (authentication),
# and a valid token that fails the CEL with 403 (authorization).
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
verdict() { case "$1" in 200) echo "$1  ✓ allowed";; 401) echo "$1  ✗ no token";; 403) echo "$1  ✗ not admin";; *) echo "$1";; esac; }
row() { printf "    %-30s %s\n" "$1" "$(verdict $2)"; }
echo "  ══ JWT at the waypoint  ·  any valid token may GET, only admin may DELETE ══"
row "no token      GET    /pets"    "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
row "alice         GET    /pets"    "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
row "alice (user)  DELETE /pets/1"  "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
row "bob (admin)   DELETE /pets/1"  "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"

**Look at the Graph now — the denials are RED.** An L7 denial is a real HTTP response, so the Graph can finally draw a block: red edges from every (token-less) client loop into `petshop-waypoint`. Contrast the L4 sections, where a denial is a reset connection and a block only ever showed as an edge going quiet. L4 deny = silence; L7 deny = red edge with a status code.

## 3.3 · Route at the waypoint: canary and header shift

**What we're doing:** show the same waypoint that checks the JWT also does traffic management, and that policy and routing compose.

**How:** attach an `HTTPRoute` whose parent is the petstore **Service** (the GAMMA pattern). Callers keep calling `petstore:8080`; the waypoint enforces a 90/10 canary to `petstore-v2` and a header shift, so `x-beta: true` goes straight to v2. The JWT policy from §3.2 still gates every request.

**What you'll see:** roughly 90/10 across v1 and v2 with a valid token, the header pinning v2, and no token still `401` on every path.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# petstore-v2: the same app (it reports served_by), the SAME ServiceAccount
# (routing does not change identity), its own Service.
kubectl --context $CTX apply -f yaml/50-l7/30-petstore-v2.yaml
kubectl --context $CTX -n petshop rollout status deploy/petstore-v2 --timeout=120s

kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: petstore-split
  namespace: petshop
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: petstore
      port: 8080
  rules:
    - matches:
        - headers:
            - name: x-beta
              value: "true"
      backendRefs:
        - name: petstore-v2
          port: 8080
    - backendRefs:
        - name: petstore
          port: 8080
          weight: 90
        - name: petstore-v2
          port: 8080
          weight: 10
EOF
kubectl --context $CTX -n petshop get httproute petstore-split \
  -o jsonpath='{.status.parents[0].conditions[?(@.type=="Accepted")].status}{" — accepted by Service/"}{.status.parents[0].parentRef.name}'; echo
sleep 10

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# the proof: distribution, header shift, and the JWT policy still in force
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "  ══ 20 GETs with alice's token  ·  the canary split (target 90/10) ══"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 20); do curl -s -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' \
  | sort | uniq -c | awk '{printf "    %-12s %2d / 20   (%d%%)\n", $2, $1, $1*5}'

echo
echo "  ══ 5 GETs with header x-beta: true  ·  pinned to v2 ══"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 5); do curl -s -m5 -H 'Authorization: Bearer $ALICE' -H 'x-beta: true' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' \
  | sort | uniq -c | awk '{printf "    %-12s %2d / 5\n", $2, $1}'

echo
NOTOK=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'x-beta: true' http://petstore:8080/pets")
echo "  no token, even on the beta route  →  $NOTOK   (JWT policy still gates every path)"

## 3.4 · Rate limit by workload identity

**What we're doing:** rate-limit one workload without touching another, even when both carry the same user's token. The user's token says who the **user** is; the certificate says who the **workload** is, and here they meet.

**How:** a rate limit whose CEL condition keys on `source.identity.serviceAccount`, the identity ztunnel proved at L4, enforced locally in the waypoint.

**What you'll see:** `storefront` allowed five times then `429`, while `checkout` presenting the same user JWT is untouched. The budget followed the workload's certificate, not the user's token.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# 5 requests/minute for the storefront IDENTITY, everyone else unlimited
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-ratelimit-storefront, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    rateLimit:
      conditional:
        - condition: 'source.identity.serviceAccount == "storefront"'
          policy:
            local:
              - requests: 5
                unit: Minutes
EOF
sleep 8

KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "  ══ 8 rapid GETs, same alice token, from two different workloads ══"
printf "    %-16s " "storefront"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo
printf "    %-16s " "checkout-blue"
kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo
echo "    limit followed the workload's certificate (storefront), not the shared user token"

`storefront: 200 ×5 then 429 429 429` — `checkout-blue: 200 ×8`, on the **same token**. The rate budget followed the workload's certificate, not the user's JWT. That is the identity thesis of this whole part, applied to L7 traffic management.